In [84]:
%cd /content
import os
if os.path.isdir("deepfake"):
    %cd deepfake
    !git pull
else:
    !git clone https://github.com/satyagalla/deepfake.git
    %cd deepfake

/content
/content/deepfake
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 7 (delta 5), reused 7 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 1.11 KiB | 378.00 KiB/s, done.
From https://github.com/satyagalla/deepfake
   9c4a52d..eb80f81  main       -> origin/main
Updating 9c4a52d..eb80f81
Fast-forward
 config.py                       | 6 ++++++
 data/selfgen_gemini_generate.py | 8 ++++++--
 2 files changed, 12 insertions(+), 2 deletions(-)


In [85]:
%pwd

'/content/deepfake'

In [19]:
!pip install -q -r requirements.txt

In [3]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
# A100 expected per build plan Environment section.

CUDA available: False
GPU: none


In [4]:
# Mount Drive for raw/checkpoint/feature persistence across runtime recycles.
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['DEEPFAKE_DATA_ROOT'] = '/content/drive/MyDrive/deepfake'

Mounted at /content/drive


In [7]:
from google.colab import userdata
import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

In [9]:
!python -c "from data.selfgen_generate import load_or_create_prompts; load_or_create_prompts(130, 20)"

[prompts] wrote new manifest to /content/drive/MyDrive/deepfake/data_raw/selfgen_intake/_prompts_manifest.json (130 in-domain, 20 off-domain)
Fatal Python error: PyGILState_Release: thread state 0x7d11f475e320 must be current when releasing
Python runtime state: finalizing (tstate=0x0000000000b8a5b0)

Thread 0x00007d12cb54b000 (most recent call first):
  <no Python frame>

Extension modules: numpy.core._multiarray_umath, numpy.core._multiarray_tests, numpy.linalg._umath_linalg, numpy.fft._pocketfft_internal, numpy.random._common, numpy.random.bit_generator, numpy.random._bounded_integers, numpy.random._mt19937, numpy.random.mtrand, numpy.random._philox, numpy.random._pcg64, numpy.random._sfc64, numpy.random._generator, torch._C, torch._C._fft, torch._C._linalg, torch._C._nested, torch._C._nn, torch._C._sparse, torch._C._special, zstandard.backend_c, pyarrow.lib, pandas._libs.tslibs.ccalendar, pandas._libs.tslibs.np_datetime, pandas._libs.tslibs.dtypes, pandas._libs.tslibs.base, pandas.

In [20]:
import json
d = json.load(open('/content/drive/MyDrive/deepfake/data_raw/selfgen_intake/_prompts_manifest.json'))
print('indomain:', len(d['indomain']), 'unique:', len(set(d['indomain'])))
print('offdomain:', len(d['offdomain']), 'unique:', len(set(d['offdomain'])))
print('any empty indomain:', any(not p.strip() for p in d['indomain']))

indomain: 130 unique: 130
offdomain: 20 unique: 20
any empty indomain: False


In [21]:
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [150]:
!python data/selfgen_gemini_generate.py > gemini.log 2>&1 &
!wait

In [179]:
!tail -f gemini.log

[prompts] reusing existing manifest at /content/drive/MyDrive/deepfake/data_raw/selfgen_intake/_prompts_manifest.json (130 in-domain, 20 off-domain)
[gemini/indomain] 0127 attempt 1/5 failed: no image returned (finish_reason: FinishReason.PROHIBITED_CONTENT) -- retrying in 1s
[gemini/indomain] 0127 attempt 2/5 failed: no image returned (finish_reason: FinishReason.PROHIBITED_CONTENT) -- retrying in 2s
[gemini/indomain] 0127 attempt 3/5 failed: no image returned (finish_reason: FinishReason.PROHIBITED_CONTENT) -- retrying in 4s
[gemini/indomain] 0127 attempt 4/5 failed: no image returned (finish_reason: FinishReason.PROHIBITED_CONTENT) -- retrying in 8s
[gemini/indomain] 0127 FAILED after 5 attempts: no image returned (finish_reason: FinishReason.PROHIBITED_CONTENT)

[gemini] done. generated=0 skipped(already done)=149 failed=1
[gemini] 1 prompt(s) failed permanently -- see /content/drive/MyDrive/deepfake/data_raw/selfgen_intake/gemini/_failures.log. Re-run this script to retry just tho

In [116]:
!python data/selfgen_gptimage_generate.py > gptimage.log 2>&1 &
!wait

In [193]:
!tail -f gptimage.log

^C


In [205]:
!find /content/drive/MyDrive/deepfake/data_raw/selfgen_intake -name "*.png" | wc -l
!ls /content/drive/MyDrive/deepfake/data_raw/selfgen_intake/gemini/offdomain/api | wc -l
!ls /content/drive/MyDrive/deepfake/data_raw/selfgen_intake/gptimage/offdomain/api | wc -l

299
20
20


In [207]:
!ps aux | grep gemini

root       55286  0.0  0.0   7372  3356 ?        S    02:57   0:00 /bin/bash -c ps aux | grep gemini
root       55288  0.0  0.0   6480  2420 ?        S    02:57   0:00 grep gemini


In [206]:
# !pkill -f selfgen_gptimage_generate.py
!ps aux | grep gptimage

root       55275  0.0  0.0   7372  3552 ?        S    02:57   0:00 /bin/bash -c ps aux | grep gptimage
root       55277  0.0  0.0   6480  2512 ?        S    02:57   0:00 grep gptimage
